In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, to_timestamp, explode
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, LongType, DoubleType

In [2]:
spark = SparkSession.builder \
    .appName("inspect-bronze") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio_admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio_password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .getOrCreate()

In [3]:
df = spark.read.parquet("s3a://bronze/coingecko/markets/")

In [4]:
market_schema = ArrayType(
    StructType([
        StructField("id", StringType()),
        StructField("symbol", StringType()),
        StructField("current_price", DoubleType()),
        StructField("market_cap", DoubleType()),
        StructField("total_volume", DoubleType()),
        StructField("market_cap_rank", LongType())
    ])
)

In [6]:
df_parsed = df.withColumn(
    "markets",
    from_json(col("bronze_source_data"), market_schema)
)

In [7]:
df_exploded = df_parsed.withColumn(
    "asset",
    explode("markets")
)

In [8]:
df_silver = df_exploded.select(
    col("asset.id").alias("asset_id"),
    col("asset.symbol"),

    col("asset.current_price"),
    col("asset.market_cap"),
    col("asset.total_volume"),
    col("asset.market_cap_rank"),

    col("ingestion_ts")
)

In [13]:
df_silver = df_silver.withColumn(
    "date",
    to_timestamp(col("ingestion_ts"), "yyyyMMdd_HH")
)

In [14]:
df_silver.show()

+------------+----------+-------------+-----------------+---------------+---------------+------------+-------------------+
|    asset_id|    symbol|current_price|       market_cap|   total_volume|market_cap_rank|ingestion_ts|               date|
+------------+----------+-------------+-----------------+---------------+---------------+------------+-------------------+
|     bitcoin|       btc|      76692.0|1.536723687223E12|3.8220818023E10|              1| 20260519_13|2026-05-19 13:00:00|
|    ethereum|       eth|      2110.24| 2.54795254781E11|1.6637335164E10|              2| 20260519_13|2026-05-19 13:00:00|
|      tether|      usdt|     0.999105| 1.89688260613E11|6.1549977946E10|              3| 20260519_13|2026-05-19 13:00:00|
| binancecoin|       bnb|       639.26|  8.6178561935E10|   7.31119759E8|              4| 20260519_13|2026-05-19 13:00:00|
|      ripple|       xrp|         1.37|  8.4989055735E10|  1.804554932E9|              5| 20260519_13|2026-05-19 13:00:00|
|    usd-coin|  